# Data Cleanup for D8 Startup Dataset

This notebook creates a **compact research subset** from `data/Startup_Data.csv`. The target is to keep **at most 20 columns total**, not 90+. The retained set is designed to cover the project's main hypotheses and a small number of controls for funding, founder quality, team structure, business model, and market traction.


## Retention logic

The final subset keeps 20 columns in total:

- `Dependent-Company Status` as the outcome.
- Main hypothesis variables: `Number of Co-founders`, `B2C or B2B venture?`, `Time to 1st investment (in months)`.
- Team and traction controls: advisor count, senior leadership size, employee growth, internet activity.
- Funding controls: last funding round size, investor participation, repeat investors, top-fund presence.
- Founder quality controls: founder experience, prior startup success, university tier.
- Business model controls: linear vs non-linear and capital intensity.
- Market and cohort controls: founding year and industry investment trend.
- One technology signal: `Disruptiveness of technology`.

Everything else is removed because it is either raw text, too redundant, too noisy, too sparse, too high-cardinality for this sample size, or outside the core econometric questions.


In [ ]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("data/Startup_Data.csv")
CLEAN_PATH = Path("data/Startup_Data_research_subset.csv")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

df = pd.read_csv(RAW_PATH, encoding="latin1")
print(f"Raw shape: {df.shape}")
df.head(3)


## Standardize obvious categorical inconsistencies

The raw file mixes labels such as `YES`, `yes`, `Yes`, lowercase intensity levels, blank strings, and `No Info`. This step cleans those inconsistencies first.


In [ ]:
def normalize_string_column(series: pd.Series) -> pd.Series:
    if not (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)):
        return series

    series = series.astype("string").str.strip()
    replacements = {
        "YES": "Yes",
        "yes": "Yes",
        "NO": "No",
        "high": "High",
        "medium": "Medium",
        "low": "Low",
        "No info": "No Info",
    }
    return series.replace(replacements)


df = df.apply(normalize_string_column)

numeric_like_columns = []
for col in df.columns:
    non_missing = df[col].dropna()
    if non_missing.empty:
        continue

    converted = pd.to_numeric(
        non_missing.astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
    if converted.notna().all():
        numeric_like_columns.append(col)

for col in numeric_like_columns:
    df[col] = pd.to_numeric(
        df[col].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )

print(f"Numeric-like columns converted: {len(numeric_like_columns)}")


## Keep only the research subset

These columns are retained on purpose. This is the compact dataset that should be used for the next stage of modeling.


In [ ]:
keep_groups = {
    "Outcome": [
        "Dependent-Company Status",
    ],
    "Main hypotheses": [
        "Number of Co-founders",
        "B2C or B2B venture?",
        "Time to 1st investment (in months)",
    ],
    "Team and traction controls": [
        "Number of of advisors",
        "Team size Senior leadership",
        "Employees count MoM change",
        "Internet Activity Score",
    ],
    "Funding controls": [
        "Last round of funding received (in milionUSD)",
        "Number of Investors in Angel and or VC",
        "Number of of repeat investors",
        "Presence of a top angel or venture fund in previous round of investment",
    ],
    "Founder quality controls": [
        "Average Years of experience for founder and co founder",
        "Have been part of successful startups in the past?",
        "Degree from a Tier 1 or Tier 2 university?",
    ],
    "Business model controls": [
        "Linear or Non-linear business model",
        "Capital intensive business e.g. e-commerce, Engineering products and operations can also cause a business to be capital intensive",
    ],
    "Market and technology controls": [
        "year of founding",
        "Industry trend in investing",
        "Disruptiveness of technology",
    ],
}

keep_columns = [col for cols in keep_groups.values() for col in cols]
missing_from_source = [col for col in keep_columns if col not in df.columns]
assert not missing_from_source, f"Columns not found: {missing_from_source}"
assert len(keep_columns) <= 20, f"Too many retained columns: {len(keep_columns)}"

retained = pd.Series(
    {col: group for group, cols in keep_groups.items() for col in cols},
    name="group",
).rename_axis("column").reset_index()

print(f"Retained columns: {len(keep_columns)}")
retained


In [ ]:
df_clean = df[keep_columns].copy()

print(f"Raw shape: {df.shape}")
print(f"Research subset shape: {df_clean.shape}")
print(f"Columns removed: {df.shape[1] - df_clean.shape[1]}")
print(f"Columns retained: {df_clean.shape[1]}")


## Quick audit of the research subset

This check shows missingness and cardinality in the final retained variables so the next notebook can decide how to encode and impute them.


In [ ]:
audit = pd.DataFrame(
    {
        "dtype": df_clean.dtypes.astype(str),
        "missing_pct": (df_clean.isna().mean() * 100).round(1),
        "nunique": df_clean.nunique(dropna=True),
    }
).sort_values(["missing_pct", "nunique"], ascending=[False, False])

audit


In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved research subset to: {CLEAN_PATH}")


## Next step

The next notebook should handle modeling preprocessing only: encode binary and ordinal variables, decide how to treat `No Info` and missing values, and then estimate the compact logit specification.
